In [1]:
# CELL 1
# Install Dependencies

!pip install -q keras-tuner

In [2]:
# CELL 2
# Imports

import os
import gc
import warnings
import shutil
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras_tuner as kt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization,
    Input
)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    TensorBoard
)

from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
import tensorflow.keras.backend as K

warnings.filterwarnings('ignore')

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

2026-05-27 13:34:09.633341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779888849.888072      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779888849.958466      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779888850.515514      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779888850.515555      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779888850.515558      58 computation_placer.cc:177] computation placer alr

TF version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
# CELL 3
# Config & Paths

# Paths
DATASET_PATH  = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000'

IMG_DIR_1     = os.path.join(DATASET_PATH, 'HAM10000_images_part_1')
IMG_DIR_2     = os.path.join(DATASET_PATH, 'HAM10000_images_part_2')

METADATA_PATH = os.path.join(DATASET_PATH, 'HAM10000_metadata.csv')

OUTPUT_DIR    = '/kaggle/working/output'
TUNER_DIR     = '/kaggle/working/tuner'

# Image / Training Params
IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_CLASSES   = 7

# Bayesian Optimization Params
TUNER_EPOCHS  = 8
MAX_TRIALS    = 10
FINAL_EPOCHS  = 60

# Mixed Precision
mixed_precision.set_global_policy('mixed_float16')

# Output Directories
for d in [OUTPUT_DIR, TUNER_DIR]:
    os.makedirs(d, exist_ok=True)

print("Config OK | IMG_SIZE:", IMG_SIZE, "| BATCH:", BATCH_SIZE)

Config OK | IMG_SIZE: 224 | BATCH: 32


In [4]:
# CELL 4
# Memory Helpers

def clear_memory():

    gc.collect()
    K.clear_session()
    tf.compat.v1.reset_default_graph()

    print("Session cleared.")


def gpu_memory_growth():

    gpus = tf.config.list_physical_devices('GPU')

    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


gpu_memory_growth()

In [5]:
# CELL 5
# Load & Analyse Data

metadata = pd.read_csv(METADATA_PATH)

print("Dataset shape:", metadata.shape)
print(metadata['dx'].value_counts())

# Class Distribution Before Balancing
fig, ax = plt.subplots(figsize=(10, 4))

counts_before = metadata['dx'].value_counts()

bars = ax.bar(
    counts_before.index,
    counts_before.values,
    color=sns.color_palette('Set2', len(counts_before))
)

ax.set_title('Class Distribution Before Balancing', fontsize=14)
ax.set_xlabel('Skin Lesion Class')
ax.set_ylabel('Sample Count')

for bar, val in zip(bars, counts_before.values):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 30,
        str(val),
        ha='center',
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, 'class_distribution_before.png'),
    dpi=150
)

plt.close()

print("Saved: class_distribution_before.png")

Dataset shape: (10015, 7)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
Saved: class_distribution_before.png


In [6]:
# CELL 6
# Image Paths

def get_image_path(image_id):

    for d in [IMG_DIR_1, IMG_DIR_2]:

        p = os.path.join(d, image_id + '.jpg')

        if os.path.exists(p):
            return p

    return None


metadata['image_path'] = metadata['image_id'].apply(get_image_path)

metadata.dropna(subset=['image_path'], inplace=True)

print("Images found:", len(metadata))

Images found: 10015


In [7]:
# CELL 7
# Label Encoding

le = LabelEncoder()

metadata['label'] = le.fit_transform(metadata['dx'])

CLASS_NAMES = le.classes_.tolist()

print("Classes:", CLASS_NAMES)

# Save Label Mapping
label_map = {i: cls for i, cls in enumerate(CLASS_NAMES)}

with open(
    os.path.join(OUTPUT_DIR, 'label_mapping.json'),
    'w'
) as f:

    json.dump(label_map, f, indent=2)

Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [8]:
# CELL 8
# Smart Oversampling

TARGET_SAMPLES = 1500


def smart_oversample(df, target, seed=42):

    balanced_parts = []

    for label_val in df['label'].unique():

        subset = df[df['label'] == label_val]

        n = len(subset)

        if n >= target:

            balanced_parts.append(subset)

        else:

            factor = min(target // n + 1, 3)

            oversampled = pd.concat(
                [subset] * factor,
                ignore_index=True
            )

            oversampled = oversampled.sample(
                n=target,
                replace=True,
                random_state=seed
            )

            balanced_parts.append(oversampled)

    balanced = pd.concat(
        balanced_parts,
        ignore_index=True
    )

    return balanced.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)


balanced_df = smart_oversample(metadata, TARGET_SAMPLES)

print("Balanced dataset size:", len(balanced_df))

print(
    balanced_df['label']
    .value_counts()
    .sort_index()
)

# Class Distribution After Balancing
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in zip(

    axes,

    [metadata, balanced_df],

    [
        'Before Balancing',
        'After Balancing'
    ]
):

    counts = df['dx'].value_counts()

    if title == 'After Balancing':

        counts = (
            balanced_df['label']
            .value_counts()
            .sort_index()
        )

        counts.index = [
            CLASS_NAMES[i]
            for i in counts.index
        ]

    ax.bar(
        counts.index,
        counts.values,
        color=sns.color_palette('husl', len(counts))
    )

    ax.set_title(f'Class Distribution {title}')
    ax.set_xlabel('Class')
    ax.set_ylabel('Samples')

    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, 'class_distribution_after.png'),
    dpi=150
)

plt.close()

print("Saved: class_distribution_after.png")

Balanced dataset size: 15705
label
0    1500
1    1500
2    1500
3    1500
4    1500
5    6705
6    1500
Name: count, dtype: int64
Saved: class_distribution_after.png


In [9]:
# CELL 9
# Train / Validation / Test Split

train_val_df, test_df = train_test_split(

    balanced_df,

    test_size=0.10,

    stratify=balanced_df['label'],

    random_state=42
)

train_df, val_df = train_test_split(

    train_val_df,

    test_size=0.15,

    stratify=train_val_df['label'],

    random_state=42
)

# Convert Labels to String
for df in [train_df, val_df, test_df]:

    df['label_str'] = df['label'].astype(str)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 12013
Validation: 2121
Test: 1571


In [10]:
# CELL 10
# Compute Class Weights

class_weights_array = compute_class_weight(

    class_weight='balanced',

    classes=np.arange(NUM_CLASSES),

    y=train_df['label']
)

class_weights = dict(
    enumerate(class_weights_array)
)

print(

    "Class weights:",

    {
        CLASS_NAMES[k]: round(v, 3)

        for k, v in class_weights.items()
    }
)

Class weights: {'akiec': np.float64(1.495), 'bcc': np.float64(1.495), 'bkl': np.float64(1.496), 'df': np.float64(1.496), 'mel': np.float64(1.496), 'nv': np.float64(0.335), 'vasc': np.float64(1.496)}


In [11]:
# CELL 11
# Data Generators

train_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input,

    rotation_range=30,

    zoom_range=0.20,

    width_shift_range=0.15,

    height_shift_range=0.15,

    shear_range=0.10,

    brightness_range=[0.8, 1.2],

    horizontal_flip=True,

    vertical_flip=True,

    fill_mode='reflect'
)

val_test_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input
)

GEN_PARAMS = dict(

    x_col='image_path',

    y_col='label_str',

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode='sparse',

    shuffle=False
)

train_gen = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    **{
        **GEN_PARAMS,
        'shuffle': True
    }
)

val_gen = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    **GEN_PARAMS
)

test_gen = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    **GEN_PARAMS
)

print("Train Steps:", len(train_gen))
print("Validation Steps:", len(val_gen))

Found 12013 validated image filenames belonging to 7 classes.
Found 2121 validated image filenames belonging to 7 classes.
Found 1571 validated image filenames belonging to 7 classes.
Train Steps: 376
Validation Steps: 67


In [12]:
# CELL 12
# Focal Loss

def focal_loss(gamma=2.0, alpha=0.25):

    def loss_fn(y_true, y_pred):

        y_true = tf.cast(y_true, tf.int32)

        y_pred = tf.cast(y_pred, tf.float32)

        y_pred = tf.clip_by_value(
            y_pred,
            1e-7,
            1.0 - 1e-7
        )

        y_true_oh = tf.one_hot(
            y_true,
            depth=NUM_CLASSES
        )

        y_true_oh = tf.cast(
            y_true_oh,
            tf.float32
        )

        cross_entropy = -y_true_oh * tf.math.log(y_pred)

        weight = alpha * y_true_oh * tf.pow(
            1 - y_pred,
            gamma
        )

        fl = weight * cross_entropy

        return tf.reduce_mean(
            tf.reduce_sum(fl, axis=-1)
        )

    return loss_fn

In [13]:
# CELL 13
# Build Model

def build_model(hp):

    # Base Model
    base = DenseNet201(

        include_top=False,

        weights='imagenet',

        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Fine Tune Layers
    fine_tune_layers = hp.Int(

        'fine_tune_layers',

        min_value=50,

        max_value=400,

        step=50
    )

    for layer in base.layers[:-fine_tune_layers]:
        layer.trainable = False

    for layer in base.layers[-fine_tune_layers:]:
        layer.trainable = True

    # Classification Head
    x = base.output

    x = GlobalAveragePooling2D()(x)

    x = BatchNormalization()(x)

    dropout_rate = hp.Float(
        'dropout',
        0.3,
        0.6,
        step=0.1
    )

    x = Dropout(dropout_rate)(x)

    dense_units = hp.Choice(
        'dense_units',
        values=[256, 512]
    )

    x = Dense(
        dense_units,
        activation='relu'
    )(x)

    x = BatchNormalization()(x)

    x = Dropout(dropout_rate)(x)

    outputs = Dense(

        NUM_CLASSES,

        activation='softmax',

        dtype='float32'
    )(x)

    model = Model(base.input, outputs)

    # Compile
    lr = hp.Choice(
        'learning_rate',
        values=[1e-3, 3e-4, 1e-4]
    )

    model.compile(

        optimizer=Adam(learning_rate=lr),

        loss=focal_loss(),

        metrics=['accuracy']
    )

    return model

In [14]:
# CELL 14 — Stage 1: Bayesian Hyperparameter Search
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("  STAGE 1 — Bayesian Hyperparameter Search")
print("="*60)

# Remove stale tuner data (prevent conflicts on re-run)
if os.path.exists(TUNER_DIR):
    shutil.rmtree(TUNER_DIR)

tuner = kt.BayesianOptimization(
    hypermodel=build_model,
    objective=kt.Objective('val_accuracy', direction='max'),
    max_trials=MAX_TRIALS,
    num_initial_points=3,        # 3 random warm-up points before Bayes kicks in
    directory=TUNER_DIR,
    project_name='ham10000_bo',
    overwrite=True
)

tuner.search_space_summary()

# Stage-1 callbacks — aggressive early stop to save GPU time
stage1_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3,
                  restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, verbose=0, min_lr=1e-6)
]

tuner.search(
    train_gen,
    validation_data=val_gen,
    epochs=TUNER_EPOCHS,
    class_weight=class_weights,
    callbacks=stage1_callbacks,
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

#  Save & print best hyperparameters 
hp_summary = {
    'learning_rate':  best_hps.get('learning_rate'),
    'dropout':        best_hps.get('dropout'),
    'dense_units':    best_hps.get('dense_units'),
    'fine_tune_layers': best_hps.get('fine_tune_layers')
}
print("\nBest Hyperparameters:")
for k, v in hp_summary.items():
    print(f"  {k}: {v}")

with open(os.path.join(OUTPUT_DIR, 'best_hyperparameters.json'), 'w') as f:
    json.dump(hp_summary, f, indent=2)

# Save all trial results
trials_summary = []
for trial in tuner.oracle.trials.values():
    trials_summary.append({
        'trial_id':    trial.trial_id,
        'score':       trial.score,
        **{k: v[0] if isinstance(v, list) else v
           for k, v in trial.hyperparameters.values.items()}
    })
pd.DataFrame(trials_summary).sort_values('score', ascending=False)\
  .to_csv(os.path.join(OUTPUT_DIR, 'tuning_history.csv'), index=False)
print("Saved: best_hyperparameters.json | tuning_history.csv")


Trial 10 Complete [00h 40m 18s]
val_accuracy: 0.8571428656578064

Best val_accuracy So Far: 0.8887317180633545
Total elapsed time: 06h 59m 38s

Best Hyperparameters:
  learning_rate: 0.001
  dropout: 0.4
  dense_units: 256
  fine_tune_layers: 150
Saved: best_hyperparameters.json | tuning_history.csv


In [ ]:
# CELL 14B — Save Best Trial Checkpoint
import shutil

# Get best trial object
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
best_trial_id = best_trial.trial_id
print(f"Best Trial ID: {best_trial_id} | Score: {best_trial.score:.4f}")

# Path where keras-tuner saved that trial's weights
best_trial_src = os.path.join(TUNER_DIR, 'ham10000_bo', 'trial_' + best_trial_id)

# Destination — clearly named best_trial folder
best_trial_dst = os.path.join(TUNER_DIR, 'best_trial')

# Remove old best_trial folder if exists (clean save)
if os.path.exists(best_trial_dst):
    shutil.rmtree(best_trial_dst)

# Copy best trial folder to best_trial/
shutil.copytree(best_trial_src, best_trial_dst)

print(f"Best trial checkpoint saved to: {best_trial_dst}")
print("Contents:")
for f in os.listdir(best_trial_dst):
    print(f"  {f}")

In [ ]:
# CELL 15 — Stage 2: Full Training with Best HPs
clear_memory()


print("  STAGE 2 — Final Deep Training")

# Strategy: mirror over two T4 GPUs
strategy = tf.distribute.MirroredStrategy()
print("Replicas (GPUs):", strategy.num_replicas_in_sync)

with strategy.scope():
    final_model = build_model(best_hps)

final_model.summary()

MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'best_model.keras')

stage2_callbacks = [
    EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=4,
        verbose=1, min_lr=1e-7
    ),
    ModelCheckpoint(
        MODEL_SAVE_PATH, save_best_only=True,
        monitor='val_accuracy', mode='max', verbose=1
    )
]

history = final_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=FINAL_EPOCHS,
    class_weight=class_weights,
    callbacks=stage2_callbacks,
    verbose=1
)

print("\nFinal training complete.")


Session cleared.

  STAGE 2 — Final Deep Training
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Replicas (GPUs): 2


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 18,824,263 (71.81 MB)

 Trainable params: 5,561,927 (21.22 MB)

 Non-trainable params: 13,262,336 (50.59 MB)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1779914156.559027     132 cuda_dnn.cc:529] Loaded cuDNN version 91002


376/376 ━━━━━━━━━━━━━━━━━━━━ 0s 733ms/step - accuracy: 0.5566 - loss: 0.2797INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

Epoch 1: val_accuracy improved from None to 0.73456, saving model to /kaggle/working/output/best_model.keras

Epoch 1: finished saving model to /kaggle/working/output/best_model.keras
376/376 ━━━━━━━━━━━━━━━━━━━━ 385s 851ms/step - accuracy: 0.6344 - loss: 0.2014 - val_accuracy: 0.7346 - val_loss: 0.2078 - learning_rate: 0.0010
Epoch 2/60
376/376 ━━━━━━━━━━━━━━━━━━━━ 0s 750ms/step - accuracy: 0.7293 - loss: 0.1123
Epoch 2: val_accuracy improved from 0.73456 to 0.76756, saving model to /kaggle/working/output/best_model.keras

Epoch 2: finished saving model to /kaggle/working/output/best_model.keras
376/376 ━━━━━━━━━━━━━━━━━━━━ 308s 819ms/step - accuracy: 0.7344 - loss: 0.1073 - val_accuracy: 0.7676 - val_loss: 0.1686 - learning_rate: 0.0010
Epoch 3/60
376/376 ━━━━━━━━━━━━━━━

In [ ]:
# CELL 16 — Training Curves
def plot_training_history(history, save_dir):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Plot 1 — Train vs Val Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train Accuracy', lw=2, color='steelblue')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   lw=2, color='orange', ls='--')
    axes[0].set_title('Train vs Val Accuracy', fontweight='bold', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot 2 — Train vs Val Loss
    axes[1].plot(history.history['loss'],     label='Train Loss', lw=2, color='steelblue')
    axes[1].plot(history.history['val_loss'], label='Val Loss',   lw=2, color='orange', ls='--')
    axes[1].set_title('Train vs Val Loss', fontweight='bold', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Plot 3 — Validation Accuracy vs Epoch (dedicated)
    val_acc = history.history['val_accuracy']
    epochs  = range(1, len(val_acc) + 1)
    best_epoch = val_acc.index(max(val_acc)) + 1
    best_val   = max(val_acc)

    axes[2].plot(epochs, val_acc, lw=2, color='green', marker='o',
                 markersize=4, label='Val Accuracy')
    axes[2].axvline(x=best_epoch, color='red', ls='--', lw=1.5,
                    label=f'Best Epoch: {best_epoch}')
    axes[2].axhline(y=best_val, color='purple', ls=':', lw=1.5,
                    label=f'Best Val Acc: {best_val:.4f}')
    axes[2].scatter([best_epoch], [best_val], color='red',
                    zorder=5, s=100)
    axes[2].set_title('Validation Accuracy vs Epoch', fontweight='bold', fontsize=13)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Validation Accuracy')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()

    # Save combined
    plt.savefig(os.path.join(save_dir, 'training_curves.png'), dpi=150)
    plt.close()
    print("Saved: training_curves.png")

    # Save val accuracy graph separately as well
    fig2, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, val_acc, lw=2, color='green', marker='o',
            markersize=4, label='Val Accuracy')
    ax.axvline(x=best_epoch, color='red', ls='--', lw=1.5,
               label=f'Best Epoch: {best_epoch}')
    ax.axhline(y=best_val, color='purple', ls=':', lw=1.5,
               label=f'Best Val Acc: {best_val:.4f}')
    ax.scatter([best_epoch], [best_val], color='red', zorder=5, s=100)
    ax.set_title('Validation Accuracy vs Epoch', fontweight='bold', fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'val_accuracy_vs_epoch.png'), dpi=150)
    plt.close()
    print("Saved: val_accuracy_vs_epoch.png")

plot_training_history(history, OUTPUT_DIR)

In [ ]:
# CELL 17 — Test Evaluation
# ─────────────────────────────────────────────
test_gen.reset()
test_loss, test_acc = final_model.evaluate(test_gen, verbose=1)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

test_gen.reset()
y_pred_proba = final_model.predict(test_gen, verbose=1)
y_pred  = np.argmax(y_pred_proba, axis=1)
y_true  = test_df['label'].values[:len(y_pred)]

# ── Classification report ─────────────────────────────────
report = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4
)
print("\nClassification Report:\n", report)

with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(f"Test Loss:     {test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.4f}\n\n")
    f.write(report)

# ── Test metrics JSON ─────────────────────────────────────
test_metrics = {'test_loss': round(float(test_loss), 4),
                'test_accuracy': round(float(test_acc), 4)}
with open(os.path.join(OUTPUT_DIR, 'test_metrics.json'), 'w') as f:
    json.dump(test_metrics, f, indent=2)
print("Saved: classification_report.txt | test_metrics.json")



In [ ]:
# CELL 18 — Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.close()
print("Saved: confusion_matrix.png")


In [ ]:
# CELL 19 — ROC-AUC Curves (One-vs-Rest)
y_true_oh = to_categorical(y_true, num_classes=NUM_CLASSES)

fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette('tab10', NUM_CLASSES)
roc_data = {}

for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_oh[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    roc_data[cls] = round(roc_auc, 4)
    ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})',
            color=color, lw=1.8)

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random')
ax.set_title('ROC-AUC Curves — Per Class (OvR)', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_auc_curves.png'), dpi=150)
plt.close()

with open(os.path.join(OUTPUT_DIR, 'roc_auc_scores.json'), 'w') as f:
    json.dump(roc_data, f, indent=2)

print("Saved: roc_auc_curves.png | roc_auc_scores.json")
print("ROC-AUC per class:", roc_data)

In [ ]:
# CELL 20 — Tuning History Visualisation
tuning_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'tuning_history.csv'))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(tuning_df)), tuning_df['score'].fillna(0),
       color=sns.color_palette('coolwarm', len(tuning_df)))
ax.set_xticks(range(len(tuning_df)))
ax.set_xticklabels([f"T{i}" for i in range(len(tuning_df))], fontsize=9)
ax.set_title('Bayesian Optimization — Trial Scores', fontweight='bold')
ax.set_xlabel('Trial')
ax.set_ylabel('Val Accuracy')
ax.axhline(tuning_df['score'].max(), color='red', ls='--',
           label=f"Best: {tuning_df['score'].max():.4f}")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tuning_history.png'), dpi=150)
plt.close()
print("Saved: tuning_history.png")

In [ ]:
# CELL 21 — Save Final Model & Summary
# Model already saved via ModelCheckpoint; save again explicitly
final_model.save(MODEL_SAVE_PATH)
print(f"Model saved: {MODEL_SAVE_PATH}")

# Final summary print

print("   Output Summary")

outputs = list(Path(OUTPUT_DIR).iterdir())
for f in sorted(outputs):
    size = os.path.getsize(f) / 1024
    print(f"  {f.name:<45} {size:>8.1f} KB")

print("\nAll outputs saved to:", OUTPUT_DIR)
